In [1]:
# ==========================================================
# Notebook 05: Grounded RAG Pipeline
# ==========================================================

import faiss
import pickle
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

import torch

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
chunks_df = pd.read_csv("data/processed/document_chunks.csv")

chunks_df.head()

,chunk_id,source_document,page_number,chunk_text,chunk_length
0,1,annual_report.pdf,1,ABBOTT INDIA LIMITED ANNUAL REPORT 2024-25 TOG...,68
1,2,annual_report.pdf,2,Abbott India Limited CONTENTS Company Overview...,500
2,3,annual_report.pdf,2,of Directors 14 Independent Auditor’s Report 1...,494
3,4,annual_report.pdf,2,34 get and stay healthy.” Munir Shaikh Wellnes...,348
4,5,annual_report.pdf,2,. Wellness for Environment 54 We cannot guaran...,470


In [3]:
faiss_index = faiss.read_index("data/embeddings/faiss_index.bin")

print("Indexed Vectors:", faiss_index.ntotal)

Indexed Vectors: 1804


In [4]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded.")

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: fe263f25-f81c-4842-ab72-e30225ad4a4e)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: fc7417cb-eab6-4caa-ae9b-0ef3b1e39b2d)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 2s [Retry 2/5].
d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=

Embedding model loaded.


In [5]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
)

generator = pipeline(
    task="text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    temperature=0.1,
    return_full_text=False,
)

print("Phi-3 Mini loaded.")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device device because they were offloaded to the cpu and disk.


Phi-3 Mini loaded.


In [6]:
def retrieve_documents(query, top_k=3):

    query_embedding = embedding_model.encode(query)

    query_embedding = np.array([query_embedding]).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(query_embedding, top_k)

    results = chunks_df.iloc[indices[0]].copy()

    results["similarity_score"] = scores[0]

    return results

In [7]:
def build_prompt(query, retrieved_docs):

    context = "\n\n".join(retrieved_docs["chunk_text"].tolist())

    prompt = f"""
<|system|>
You are a helpful AI assistant.

Answer ONLY using the information provided
in the context below.

If the answer cannot be found,
say:
"The information was not found in the document."

<|user|>

Context:
{context}

Question:
{query}

<|assistant|>
"""

    return prompt

In [8]:
query = "What was the company's revenue?"

retrieved_docs = retrieve_documents(query)

prompt = build_prompt(query, retrieved_docs)

print(prompt)


<|system|>
You are a helpful AI assistant.

Answer ONLY using the information provided
in the context below.

If the answer cannot be found,
say:
"The information was not found in the document."

<|user|>

Context:
. Opening Stock Finished goods 108.21 126.67 (c) Reconciling the amount of revenue recognised in the Statement of Profit and Loss with the contracted price : Stock-in-trade 375.94 367.69 For the year ended For the year ended Work-in-progress 17.76 15.28 March 31, 2025 March 31, 2024 Less : Closing Stock Revenue as per contracted price 6,536.15 5,974.49 Finished goods (119.16) (108.21) Add/(Less) : Adjustments Stock-in-trade (543.00) (375.94) - Sales Return (154.11) (168.31) Work-in-progress

- Sales Return (154.11) (168.31) Work-in-progress (20.43) (17.76) - Discounts (36.64) (26.35) (180.68) 7.73 Net revenue from sale of products and rendering of services 6,345.40 5,779.83 28 EMPLOYEE BENEFITS EXPENSE Information about the Company’s performance obligations are summarized b

In [9]:
def local_llm(prompt):

    response = generator(prompt)

    answer = response[0]["generated_text"]

    return answer.strip()

In [10]:
def grounded_rag(query, top_k=3):

    # -----------------------------------
    # Step 1: Retrieve Chunks
    # -----------------------------------

    retrieved_docs = retrieve_documents(query=query, top_k=top_k)

    # -----------------------------------
    # Step 2: Build Prompt
    # -----------------------------------

    prompt = build_prompt(query=query, retrieved_docs=retrieved_docs)

    # -----------------------------------
    # Step 3: Generate Answer
    # -----------------------------------

    answer = local_llm(prompt)

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "prompt": prompt,
        "answer": answer,
    }

In [11]:
result = grounded_rag(query="What was the company's revenue?", top_k=3)

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
You are not running the flash-attention implementation, expect numerical differences.


In [14]:
# result = grounded_rag(query="What was the CEO's personal salary?", top_k=3)

# print(result["answer"])